In [ ]:
import pandas as pd
from datetime import date, timedelta
from datetime import datetime
import os
import shutil
import pathlib
from nbconvert import PythonExporter
import polars as pl
import glob

In [14]:
notebook_dict = {
  'IEX.ipynb': 1,
  'RTA.ipynb': 1,
}

def convert_notebook(notebook_path):
    exporter = PythonExporter()
    output, _ = exporter.from_filename(notebook_path)
    return output

first_glob = os.path.expanduser("~").replace("\\", "/")
print(f"Currently using base directory: {first_glob}")

Currently using base directory: C:/Users/ADMIN


In [15]:
for nbook in notebook_dict:
    if notebook_dict[nbook] == 0:
        print(f"Not executing notebook {nbook}.")
        continue
        
    nbook_path = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Python_Code/{nbook}'
    
    if os.path.isfile(nbook_path):
        print(f"Running notebook: {nbook}")
        try:
            script = convert_notebook(nbook_path)
            exec(script)
        except Exception as e:
            print(f"Error running notebook: {nbook}")
            print(str(e))
    else:
        print(f"Error: Notebook {nbook} not found.")

Running notebook: IEX.ipynb


Could not determine dtype for column 7, falling back to string
Could not determine dtype for column 8, falling back to string
Could not determine dtype for column 7, falling back to string
Could not determine dtype for column 8, falling back to string
Could not determine dtype for column 66, falling back to string
Could not determine dtype for column 67, falling back to string
Could not determine dtype for column 68, falling back to string
Could not determine dtype for column 66, falling back to string
Could not determine dtype for column 67, falling back to string
Could not determine dtype for column 68, falling back to string
Could not determine dtype for column 47, falling back to string
Could not determine dtype for column 49, falling back to string
Could not determine dtype for column 66, falling back to string
Could not determine dtype for column 67, falling back to string
Could not determine dtype for column 68, falling back to string
Could not determine dtype for column 66, fal

Running notebook: RTA.ipynb


Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43,

shape: (5, 4)
┌────────┬───────────┬──────────────────────────┬────────────────────┐
│ Month  ┆ EID       ┆ employee_name            ┆ count_working_days │
│ ---    ┆ ---       ┆ ---                      ┆ ---                │
│ str    ┆ i64       ┆ str                      ┆ i32                │
╞════════╪═══════════╪══════════════════════════╪════════════════════╡
│ Apr-24 ┆ 101957833 ┆ DUC ANH  NGUYEN          ┆ 22                 │
│ Apr-24 ┆ 101988327 ┆ TRONG TIEN  TRINH        ┆ 22                 │
│ Apr-24 ┆ 101993770 ┆ THUY TIEN  NGUYEN        ┆ 20                 │
│ Apr-24 ┆ 101993774 ┆ HIEU THI TRAN            ┆ 19                 │
│ Apr-24 ┆ 102029874 ┆ THIEN THANH TOAN  TRUONG ┆ 21                 │
└────────┴───────────┴──────────────────────────┴────────────────────┘
['After Call Work', 'Available Chat', 'Available Phone', 'Break', 'Coaching', 'End of Shift', 'Login', 'Lunch', 'Not Ready', 'Offline', 'Outbound Call', 'Phone Talking', 'Restroom', 'System Issue', 'Team M

In [21]:
def input_data(folder_path, sheet_name=None):
    file_paths = glob.glob(f"{folder_path}/*.xlsx") + glob.glob(f"{folder_path}/*.csv")
    df_list = []

    for file in file_paths:
        if file.endswith('.xlsx'):
            df = pl.read_excel(file, sheet_name=sheet_name, infer_schema_length=0)
        elif file.endswith('.csv'):
            try:
                df = pl.read_csv(file, encoding="utf-8", infer_schema_length=0)
            except:
                df = pl.read_csv(file, encoding="ISO-8859-1", ignore_errors=True, infer_schema_length=0)
        
        df = df.with_columns([
            pl.col(col).cast(pl.String) 
            for col in df.columns
        ])
        
        df_list.append(df)
    
    merged_df = pl.concat(df_list, how='vertical')

    return merged_df

In [22]:
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(100)

def filter_shift_discrepancies(df_input) -> pl.DataFrame:
    if isinstance(df_input, pd.DataFrame):
        df = pl.from_pandas(df_input)
    else:
        df = df_input

    hc_mismatch_condition = (
        pl.col("hc_actual").cast(pl.Float64, strict=False).fill_null(0.0) != 
        pl.col("hc_present").cast(pl.Float64, strict=False).fill_null(0.0)
    )

    lob_condition = pl.col("LOB") != "Support"

    shift_condition = pl.col("First Shift") != "Training"

    result = df.filter(hc_mismatch_condition & lob_condition & shift_condition)
    
    if "Date_Converted" in result.columns:
        result = result.sort("Date_Converted")

    return result

RTA_REPORT_GENERATE = input_data(f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_RTA')
print(RTA_REPORT_GENERATE.columns)

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.col('Date_Converted').str.slice(0, 10).str.strptime(pl.Date, format='%Y-%m-%d'),
    pl.col('Target').cast(pl.Float64), 
    pl.col('duration').cast(pl.Float64),
    pl.col('sum_productive').cast(pl.Float64),
    pl.col('start').str.strip_chars().str.to_datetime("%Y-%m-%d %H:%M:%S%.3f", strict=False),
    pl.col('end').str.strip_chars().str.to_datetime("%Y-%m-%d %H:%M:%S%.3f", strict=False)
])

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.filter(pl.col('duration').is_not_null())
RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.col('Shift Tracking').cast(pl.Utf8).str.strip_chars(),
    pl.col('Open Time').cast(pl.Float64).fill_null(0.0), 
    pl.col('Extra Time').cast(pl.Float64).fill_null(0.0)    
])
 
RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.when(pl.col('Shift Tracking') == 'HDL').then(0.5)
    .when(
        (pl.col('Shift Tracking').is_in(['PR','PR - OT','PO'])) & 
        ((pl.col('Open Time') > 0) | (pl.col('Extra Time') > 0))
    ).then(1.0)
    .when(
        (pl.col('Shift Tracking').is_in([
            'Training Offline', 'Sick Leave', 'Paid Leave', 'Billable Training'
        ])) & (pl.col('Open Time') == 0)
    ).then(1.0)
    .when(
        (pl.col('Shift Tracking') != 'HDL') & 
        (pl.col('Shift Tracking') != 'PR') & 
        (pl.col('Open Time') == 0)
    ).then(0.0)
    .otherwise(None)
    .alias('hc_present')
])
 
filtered_data = RTA_REPORT_GENERATE.select([
    'Date_Converted', 'Employee Name', 'LOB', 'Detail Status','IEX ID','First Shift','Shift Tracking', 
    'start', 'end','Target', 'duration', 'sum_productive','training-idle','Extra Time', 'hc_schedule',
    'hc_actual', 'hc_present'
])
filtered_data = filtered_data.with_columns([
    pl.col('hc_actual').cast(pl.Float64),
    pl.col('hc_present').cast(pl.Float64)
])
 
start_date = '2026-04-01'
end_date = '2026-04-16'
 
start_date_parsed = pl.lit(start_date).str.strptime(pl.Date, format='%Y-%m-%d')
end_date_parsed = pl.lit(end_date).str.strptime(pl.Date, format='%Y-%m-%d')
 
sorted_data = filtered_data.sort('Date_Converted')
filtered_data = sorted_data.filter((pl.col('Date_Converted') >= start_date_parsed) & (pl.col('Date_Converted') <= end_date_parsed))
 
filtered_data_pd = filtered_data.to_pandas()

filtered_data_1 = filter_shift_discrepancies(filtered_data)
filtered_data_1

['Year', 'Month', 'Week_Monday', 'Date_Converted', 'Employee Name', 'Email Id', 'OracleID', 'IEX ID', 'Target', 'Datetime_Fluctuate_Start_Shift', 'Datetime_Fluctuate_End_Shift', 'Fluctuate Shift', 'Datetime_First_Start_Shift', 'Datetime_First_End_Shift', 'First Shift', 'Alias', 'LOB', 'LOB_2', 'LOB_3', 'Supervisor Name', 'Wave', 'Detail Status', 'Status', 'Time_Of_Day', 'Open Time', 'Extra Time', 'Break Time', 'Lunch Time', 'Training', 'NCNS', 'AL', 'Unplanned', 'Planned', 'Roster Presented', 'Roster Scheduled', 'Night_Shift', 'Shift Tracking', 'OT Type', 'Combined OT Range', 'OT PreShift', 'OT PostShift', 'start', 'end', 'total_time_chat_handle', 'sum_productive', 'break', 'lunch', 'coaching-idle', 'training-idle', 'outbound-idle', 'break_count', 'other_status', 'over_break', 'over_lunch', 'exceed_break', 'duration', 'hc_actual', 'hc_schedule', 'time_late', 'time_leave', 'lateness', 'adherence_time', 'total_time_pull_out', 'ramco_marked', 'nsa_authorize', 'ot_authorize', 'hours', 'ot_

Date_Converted,Employee Name,LOB,Detail Status,IEX ID,First Shift,Shift Tracking,start,end,Target,duration,sum_productive,training-idle,Extra Time,hc_schedule,hc_actual,hc_present
date,str,str,str,str,str,str,datetime[ms],datetime[ms],f64,f64,f64,str,f64,str,f64,f64
2026-04-02,"""PHAN QUYNH MY""","""Lodging""","""Lodging Production""","""3085329""","""1830-2200""","""PO""",2026-04-02 18:29:21,2026-04-02 22:00:42,3.5,3.522491,3.502085,"""0""",3.5,"""0.5""",0.5,1.0
2026-04-02,"""VUU TIEN NHAN""","""Lodging""","""Lodging Production""","""3089155""","""Off""","""Off""",2026-04-02 18:13:54,2026-04-02 21:31:49,0.0,3.298494,3.016797,"""0""",0.0,"""0""",0.5,0.0
2026-04-03,"""BUI HOANG CAM NHUNG""","""Lodging""","""Lodging Production""","""3097227""","""0430-1130""","""PO""",2026-04-03 04:25:39,2026-04-03 11:19:35,6.75,6.898855,6.822499,"""0""",6.75,"""1""",0.5,1.0
2026-04-03,"""TRAN THI KIEU""","""Non_Lodging""","""Non Lodging Production""","""3089116""","""Extra Hours Offline""","""Extra Hours Offline""",2026-04-03 06:25:43,2026-04-03 11:29:46,0.0,5.067297,4.736204,"""0""",0.0,"""0""",0.5,0.0
2026-04-05,"""HOANG THI PHUONG HOA""","""Non_Lodging""","""Non Lodging Production""","""3000182""","""Off""","""Off""",2026-04-05 05:56:39,2026-04-05 12:02:00,0.0,6.089392,5.780169,"""0""",0.0,"""0""",0.5,0.0


In [23]:
def get_xlsx_filenames(data_dir):
    xlsx_files = []
    for filename in pathlib.Path(data_dir).glob('**/*.xlsx'):
        xlsx_files.append(filename.name)
    return xlsx_files
def create_dataframe_with_filenames(data_dir):
    xlsx_filenames = get_xlsx_filenames(data_dir)
    df = pd.DataFrame(xlsx_filenames, columns=['Filename'])
    return df
def get_latest_xlsx_files(df):
    df = df[df['Filename'].str.match(r'\d{4}-\d{2}-\d{2}\.xlsx')]
    df['Date'] = pd.to_datetime(df['Filename'].str.replace('.xlsx', ''), format='%Y-%m-%d')
    df_sorted = df.sort_values(by='Date', ascending=False)
    latest_files = df_sorted.head(8)['Filename'].tolist()
    return latest_files
def delete_all_xlsx_files(directory):
    for filename in pathlib.Path(directory).glob('*.xlsx'):
        os.remove(filename)
def copy_latest_files(source_dir, dest_dir, latest_files):
    for filename in latest_files:
        source_path = pathlib.Path(source_dir) / filename
        dest_path = pathlib.Path(dest_dir) / filename
        shutil.copy(source_path, dest_path)

source_dir = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_RTA'
dest_dir = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_RTA_FOR_REPORT'

df = create_dataframe_with_filenames(source_dir)
latest_files = get_latest_xlsx_files(df)
delete_all_xlsx_files(dest_dir)
copy_latest_files(source_dir, dest_dir, latest_files)